In [8]:
"""
Vietnamese Legal Concept & Relation Similarity Finder
Uses VoVanPhuc/sup-SimCSE-VietNamese-phobert-base for embeddings with GPU support
Stores in MongoDB and finds similar items considering synonyms
"""

import torch
from transformers import AutoModel, AutoTokenizer
from pymongo import MongoClient, UpdateOne
import numpy as np
from typing import List, Dict, Tuple, Optional
from sklearn.metrics.pairwise import cosine_similarity
from bson import ObjectId
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class VietnameseSimilarityFinder:
    def __init__(self, mongo_uri: str = "mongodb://localhost:27017/",
                 db_name: str = "legal_kb",
                 use_gpu: bool = True):
        """Initialize with MongoDB connection and SimCSE model"""
        # MongoDB setup
        self.client = MongoClient(mongo_uri)
        self.db = self.client[db_name]
        self.concepts = self.db['concepts']
        self.relations = self.db['relations']
        self.triplets = self.db['triplets']

        # GPU setup
        self.device = self._setup_device(use_gpu)
        logger.info(f"Using device: {self.device}")

        # Load Vietnamese SimCSE model
        logger.info("Loading VoVanPhuc/sup-SimCSE-VietNamese-phobert-base...")
        self.model_name = "VoVanPhuc/sup-SimCSE-VietNamese-phobert-base"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name)
        self.model.to(self.device)
        self.model.eval()
        logger.info("Model loaded successfully")

        # Create indexes for efficient searching
        self._create_indexes()

    def _setup_device(self, use_gpu: bool) -> torch.device:
        """Setup GPU or CPU device"""
        if use_gpu and torch.cuda.is_available():
            device = torch.device("cuda")
            logger.info(f"GPU available: {torch.cuda.get_device_name(0)}")
            logger.info(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        else:
            device = torch.device("cpu")
            if use_gpu and not torch.cuda.is_available():
                logger.warning("GPU requested but not available, using CPU")
        return device

    def _create_indexes(self):
        """Create MongoDB indexes for efficient queries"""
        self.concepts.create_index("name")
        self.concepts.create_index("embedding")
        self.relations.create_index("name")
        self.relations.create_index("embedding")
        self.triplets.create_index([("subject_id", 1), ("relation_id", 1), ("object_id", 1)])
        self.triplets.create_index("subject_name")
        self.triplets.create_index("object_name")
        logger.info("Indexes created successfully")

    def get_embedding(self, text: str) -> np.ndarray:
        """Generate embedding for a text using SimCSE on GPU"""
        with torch.no_grad():
            inputs = self.tokenizer(text, return_tensors="pt",
                                   padding=True, truncation=True,
                                   max_length=512)
            # Move inputs to GPU
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            outputs = self.model(**inputs)
            # Use CLS token embedding
            embedding = outputs.last_hidden_state[:, 0, :].squeeze()
            # Move back to CPU for numpy conversion
            return embedding.cpu().numpy()

    def get_batch_embeddings(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """Generate embeddings for multiple texts efficiently using GPU batching"""
        all_embeddings = []

        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i + batch_size]

                # Tokenize batch
                inputs = self.tokenizer(batch_texts, return_tensors="pt",
                                       padding=True, truncation=True,
                                       max_length=512)
                # Move to GPU
                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                outputs = self.model(**inputs)
                # Extract CLS embeddings
                embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                all_embeddings.append(embeddings)

        return np.vstack(all_embeddings)

    def get_text_variants(self, name: str, synonyms: List[str]) -> List[str]:
        """Get all text variants including name and synonyms"""
        variants = [name]
        if synonyms:
            variants.extend(synonyms)
        return variants

    def get_combined_embedding(self, name: str, synonyms: List[str]) -> np.ndarray:
        """
        Get combined embedding considering name and all synonyms

        Strategy: Average pooling of all variant embeddings

        Why this works:
        1. Each variant (name + synonyms) gets its own embedding
        2. Averaging captures the semantic center of all meanings
        3. Normalization ensures consistent similarity calculations

        Example:
        - name: "gồm"
        - synonyms: ["bao gồm", "chứa", "có"]
        - Creates 4 embeddings, averages them
        - Result captures all semantic variations
        """
        variants = self.get_text_variants(name, synonyms)

        # Use batch processing for efficiency
        embeddings = self.get_batch_embeddings(variants, batch_size=len(variants))

        # Average all embeddings (semantic centroid)
        combined_embedding = np.mean(embeddings, axis=0)
        # Normalize to unit vector for cosine similarity
        combined_embedding = combined_embedding / np.linalg.norm(combined_embedding)

        return combined_embedding

    def generate_and_store_concept_embeddings(self, batch_size: int = 32):
        """Generate embeddings for all concepts and store in MongoDB using GPU"""
        logger.info("Generating embeddings for concepts...")

        concepts_list = list(self.concepts.find({}))
        total = len(concepts_list)

        if total == 0:
            logger.warning("No concepts found in database")
            return

        updates = []
        for idx, concept in enumerate(concepts_list, 1):
            name = concept['name']
            synonyms = concept.get('synonym', [])

            # Generate combined embedding
            embedding = self.get_combined_embedding(name, synonyms)

            # Prepare update
            updates.append(
                UpdateOne(
                    {'_id': concept['_id']},
                    {'$set': {
                        'embedding': embedding.tolist(),
                        'embedding_model': self.model_name,
                        'embedding_device': str(self.device),
                        'embedding_includes_synonyms': len(synonyms) > 0,
                        'synonym_count': len(synonyms)
                    }}
                )
            )

            # Batch update
            if len(updates) >= batch_size or idx == total:
                self.concepts.bulk_write(updates)
                updates = []
                logger.info(f"Processed {idx}/{total} concepts")

        logger.info(f"✓ Generated embeddings for {total} concepts")

    def generate_and_store_relation_embeddings(self, batch_size: int = 128):
        """Generate embeddings for all relations and store in MongoDB using GPU"""
        logger.info("Generating embeddings for relations...")

        relations_list = list(self.relations.find({}))
        total = len(relations_list)

        if total == 0:
            logger.warning("No relations found in database")
            return

        updates = []
        for idx, relation in enumerate(relations_list, 1):
            name = relation['name']
            synonyms = relation.get('synonym', [])

            # Generate combined embedding
            embedding = self.get_combined_embedding(name, synonyms)

            # Prepare update
            updates.append(
                UpdateOne(
                    {'_id': relation['_id']},
                    {'$set': {
                        'embedding': embedding.tolist(),
                        'embedding_model': self.model_name,
                        'embedding_device': str(self.device),
                        'embedding_includes_synonyms': len(synonyms) > 0,
                        'synonym_count': len(synonyms)
                    }}
                )
            )

            # Batch update
            if len(updates) >= batch_size or idx == total:
                self.relations.bulk_write(updates)
                updates = []
                logger.info(f"Processed {idx}/{total} relations")

        logger.info(f"✓ Generated embeddings for {total} relations")

    def find_similar_concepts(self, concept_name: str, top_k: int = 5,
                             include_synonyms: bool = True,
                             min_similarity: float = 0.0) -> List[Dict]:
        """
        Find similar concepts to the given concept name

        Args:
            concept_name: Name of the concept to find similar items for
            top_k: Number of similar items to return
            include_synonyms: Whether to consider synonyms in similarity
            min_similarity: Minimum similarity threshold (0.0 to 1.0)

        Returns:
            List of similar concepts with similarity scores
        """
        # Get the query concept
        query_concept = self.concepts.find_one({'name': concept_name})
        if not query_concept:
            logger.error(f"Concept '{concept_name}' not found")
            return []

        # Get or generate query embedding
        if 'embedding' in query_concept:
            query_embedding = np.array(query_concept['embedding'])
        else:
            synonyms = query_concept.get('synonym', []) if include_synonyms else []
            query_embedding = self.get_combined_embedding(concept_name, synonyms)

        # Get all concepts with embeddings
        all_concepts = list(self.concepts.find(
            {'embedding': {'$exists': True},
             '_id': {'$ne': query_concept['_id']}}
        ))

        if not all_concepts:
            logger.warning("No concepts with embeddings found")
            return []

        # Calculate similarities
        embeddings = np.array([c['embedding'] for c in all_concepts])
        similarities = cosine_similarity([query_embedding], embeddings)[0]

        # Filter by minimum similarity
        valid_indices = np.where(similarities >= min_similarity)[0]

        # Sort by similarity
        sorted_indices = valid_indices[np.argsort(similarities[valid_indices])[::-1]][:top_k]

        results = []
        for idx in sorted_indices:
            concept = all_concepts[idx]
            results.append({
                '_id': concept['_id'],
                'name': concept['name'],
                'synonyms': concept.get('synonym', []),
                'similarity_score': float(similarities[idx]),
                'documents_count': len(concept.get('documents', []))
            })

        return results

    def find_similar_relations(self, relation_name: str, top_k: int = 5,
                              include_synonyms: bool = True,
                              min_similarity: float = 0.0) -> List[Dict]:
        """
        Find similar relations to the given relation name

        Args:
            relation_name: Name of the relation to find similar items for
            top_k: Number of similar items to return
            include_synonyms: Whether to consider synonyms in similarity
            min_similarity: Minimum similarity threshold (0.0 to 1.0)

        Returns:
            List of similar relations with similarity scores
        """
        # Get the query relation
        query_relation = self.relations.find_one({'name': relation_name})
        if not query_relation:
            logger.error(f"Relation '{relation_name}' not found")
            return []

        # Get or generate query embedding
        if 'embedding' in query_relation:
            query_embedding = np.array(query_relation['embedding'])
        else:
            synonyms = query_relation.get('synonym', []) if include_synonyms else []
            query_embedding = self.get_combined_embedding(relation_name, synonyms)

        # Get all relations with embeddings
        all_relations = list(self.relations.find(
            {'embedding': {'$exists': True},
             '_id': {'$ne': query_relation['_id']}}
        ))

        if not all_relations:
            logger.warning("No relations with embeddings found")
            return []

        # Calculate similarities
        embeddings = np.array([r['embedding'] for r in all_relations])
        similarities = cosine_similarity([query_embedding], embeddings)[0]

        # Filter by minimum similarity
        valid_indices = np.where(similarities >= min_similarity)[0]

        # Sort by similarity
        sorted_indices = valid_indices[np.argsort(similarities[valid_indices])[::-1]][:top_k]

        results = []
        for idx in sorted_indices:
            relation = all_relations[idx]
            results.append({
                '_id': relation['_id'],
                'name': relation['name'],
                'synonyms': relation.get('synonym', []),
                'similarity_score': float(similarities[idx]),
                'documents_count': len(relation.get('documents', []))
            })

        return results

    def find_similar_by_id(self, item_id: str, item_type: str = 'concept',
                          top_k: int = 5, min_similarity: float = 0.0) -> List[Dict]:
        """
        Find similar items by ObjectId

        Args:
            item_id: ObjectId string
            item_type: 'concept' or 'relation'
            top_k: Number of similar items to return
            min_similarity: Minimum similarity threshold
        """
        collection = self.concepts if item_type == 'concept' else self.relations
        item = collection.find_one({'_id': ObjectId(item_id)})

        if not item:
            logger.error(f"{item_type} with id '{item_id}' not found")
            return []

        if item_type == 'concept':
            return self.find_similar_concepts(item['name'], top_k, min_similarity=min_similarity)
        else:
            return self.find_similar_relations(item['name'], top_k, min_similarity=min_similarity)

    def get_triplet_context(self, triplet_id: str) -> Dict:
        """
        Get full context for a triplet including embeddings

        Args:
            triplet_id: ObjectId string of the triplet

        Returns:
            Dict with triplet info and related embeddings
        """
        triplet = self.triplets.find_one({'_id': ObjectId(triplet_id)})
        if not triplet:
            logger.error(f"Triplet with id '{triplet_id}' not found")
            return {}

        # Get subject, relation, object details
        subject = self.concepts.find_one({'_id': triplet['subject_id']})
        relation = self.relations.find_one({'_id': triplet['relation_id']})
        obj = self.concepts.find_one({'_id': triplet['object_id']})

        return {
            'triplet': {
                '_id': str(triplet['_id']),
                'subject_name': triplet['subject_name'],
                'relation_name': triplet['relation_name'],
                'object_name': triplet['object_name'],
                'documents': triplet.get('documents', [])
            },
            'subject': {
                '_id': str(subject['_id']) if subject else None,
                'name': subject['name'] if subject else None,
                'synonyms': subject.get('synonym', []) if subject else [],
                'has_embedding': 'embedding' in subject if subject else False
            },
            'relation': {
                '_id': str(relation['_id']) if relation else None,
                'name': relation['name'] if relation else None,
                'synonyms': relation.get('synonym', []) if relation else [],
                'has_embedding': 'embedding' in relation if relation else False
            },
            'object': {
                '_id': str(obj['_id']) if obj else None,
                'name': obj['name'] if obj else None,
                'synonyms': obj.get('synonym', []) if obj else [],
                'has_embedding': 'embedding' in obj if obj else False
            }
        }

    def find_similar_triplets(self, triplet_id: str, top_k: int = 5) -> List[Dict]:
        """
        Find similar triplets based on subject, relation, and object similarity

        Args:
            triplet_id: ObjectId string of the query triplet
            top_k: Number of similar triplets to return

        Returns:
            List of similar triplets with similarity scores
        """
        # Get triplet context
        context = self.get_triplet_context(triplet_id)
        if not context:
            return []

        triplet = context['triplet']

        # Find similar subjects, relations, and objects
        similar_subjects = self.find_similar_concepts(triplet['subject_name'], top_k=10)
        similar_relations = self.find_similar_relations(triplet['relation_name'], top_k=10)
        similar_objects = self.find_similar_concepts(triplet['object_name'], top_k=10)

        # Find triplets with similar components
        candidate_triplets = []

        # Search for triplets with similar subjects
        for subj in similar_subjects[:5]:
            triplets = list(self.triplets.find({'subject_name': subj['name']}))
            for t in triplets:
                if str(t['_id']) != triplet_id:
                    candidate_triplets.append({
                        'triplet': t,
                        'subject_similarity': subj['similarity_score'],
                        'match_type': 'subject'
                    })

        # Search for triplets with similar objects
        for obj in similar_objects[:5]:
            triplets = list(self.triplets.find({'object_name': obj['name']}))
            for t in triplets:
                if str(t['_id']) != triplet_id:
                    candidate_triplets.append({
                        'triplet': t,
                        'object_similarity': obj['similarity_score'],
                        'match_type': 'object'
                    })

        # Remove duplicates and calculate combined scores
        unique_triplets = {}
        for item in candidate_triplets:
            tid = str(item['triplet']['_id'])
            if tid not in unique_triplets:
                unique_triplets[tid] = {
                    'triplet': item['triplet'],
                    'subject_similarity': 0.0,
                    'object_similarity': 0.0,
                    'relation_similarity': 0.0
                }

            if 'subject_similarity' in item:
                unique_triplets[tid]['subject_similarity'] = max(
                    unique_triplets[tid]['subject_similarity'],
                    item['subject_similarity']
                )
            if 'object_similarity' in item:
                unique_triplets[tid]['object_similarity'] = max(
                    unique_triplets[tid]['object_similarity'],
                    item['object_similarity']
                )

        # Calculate combined similarity score
        results = []
        for tid, data in unique_triplets.items():
            combined_score = (
                data['subject_similarity'] * 0.4 +
                data['relation_similarity'] * 0.2 +
                data['object_similarity'] * 0.4
            )

            results.append({
                '_id': data['triplet']['_id'],
                'subject_name': data['triplet']['subject_name'],
                'relation_name': data['triplet']['relation_name'],
                'object_name': data['triplet']['object_name'],
                'documents': data['triplet'].get('documents', []),
                'similarity_score': combined_score,
                'subject_similarity': data['subject_similarity'],
                'relation_similarity': data['relation_similarity'],
                'object_similarity': data['object_similarity']
            })

        # Sort by combined score
        results.sort(key=lambda x: x['similarity_score'], reverse=True)

        return results[:top_k]

    def cleanup(self):
        """Cleanup resources"""
        self.client.close()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# Example usage
def main():
    # Initialize the similarity finder with GPU
    finder = VietnameseSimilarityFinder(
        mongo_uri="mongodb://localhost:27017/",
        db_name="KB_PROPERTY_LAW",
        use_gpu=True  # Set to False to use CPU
    )

    try:
        # Generate and store embeddings for all concepts and relations
        print("\n=== Generating Embeddings with GPU ===")
        # finder.generate_and_store_concept_embeddings(batch_size=64)
        # finder.generate_and_store_relation_embeddings(batch_size=64)

        # Example 1: Find similar concepts
        print("\n=== Finding Similar Concepts ===")
        print("Query: 'tổ chức chính trị'")
        similar_concepts = finder.find_similar_concepts(
            "tổ chức chính trị",
            top_k=5,
            min_similarity=0.85
        )

        for i, result in enumerate(similar_concepts, 1):
            print(f"\n{i}. {result['name']}")
            print(f"   Similarity: {result['similarity_score']:.4f}")
            if result['synonyms']:
                print(f"   Synonyms: {', '.join(result['synonyms'])}")
            print(f"   Documents: {result['documents_count']}")

        # Example 2: Find similar relations
        print("\n\n=== Finding Similar Relations ===")
        print("Query: 'gồm'")
        similar_relations = finder.find_similar_relations(
            "gồm",
            top_k=5,
            min_similarity=0.9
        )
    finally:
        # Cleanup
        finder.cleanup()
        print("\n✓ Cleanup completed")


if __name__ == "__main__":
    main()

INFO:__main__:GPU available: NVIDIA GeForce RTX 3050 Laptop GPU
INFO:__main__:GPU memory: 4.29 GB
INFO:__main__:Using device: cuda
INFO:__main__:Loading VoVanPhuc/sup-SimCSE-VietNamese-phobert-base...
INFO:__main__:Model loaded successfully
INFO:__main__:Indexes created successfully



=== Generating Embeddings with GPU ===

=== Finding Similar Concepts ===
Query: 'tổ chức chính trị'

1. các tổ chức chính trị
   Similarity: 0.9395
   Documents: 1

2. đại diện tổ chức chính trị
   Similarity: 0.8930
   Documents: 1

3. chính phủ cơ quan
   Similarity: 0.8926
   Documents: 1

4. tổ chức tài chính
   Similarity: 0.8805
   Documents: 1

5. tổ chức chính trị xã hội
   Similarity: 0.8794
   Documents: 1


=== Finding Similar Relations ===
Query: 'gồm'


=== Triplet Context ===
Triplet: tổ chức trong nước -> gồm -> mặt trận tổ quốc việt nam
Documents: 1

Subject embeddings: True
Relation embeddings: True
Object embeddings: True

✓ Cleanup completed


## Identify Meaningless triplets